# BaTiO₃ Phase 02 — Scientific schema and data dictionary

**Project:** Chemistry-aware materials-data curation and AI-ready workflow engineering  
**Material system:** Sr/La-substituted barium titanate, nominally (Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃  
**Scientific stage:** Field semantics, 82-input descriptor definitions, evidence provenance and schema validation  
**Method:** Deterministic, read-only inspection of publisher workbook S1–S4 with machine-readable dictionary exports.

**Research purpose.** Transform the Phase 01 workbook audit into an explicit scientific data model: each field has an interpretable role, original worksheet location, type, evidence class, unit policy, missingness policy and review status. This is a **schema design and source-description exercise**, not DFT replication, data standardisation, independent ML training or experimental verification.

**Source:** [Peer-reviewed ceramic study](https://doi.org/10.1063/5.0314819) · [Official supplementary collection](https://doi.org/10.60893/figshare.jap.c.8234602). The article and supplementary information explain scientific methods and feature abbreviations; the Excel file is the only runtime dependency.

**Workflow commitments:** clear scientific explanations, simple standard-library code, reproducible calculations, concise setup notes, explicit error handling, evidence-aware interpretation, automated tests and actionable outputs. Outputs are local working artefacts; **publication permissions and sanitisation must be reviewed separately**.

## 1 | Scientific question and evidence boundary

BaTiO₃ is an `ABO₃` perovskite: Ba/Sr/La belong to the nominal **A-site** design space, Ti is the **B-site** element, and O is the anion sublattice. Sr²⁺ replacement of Ba²⁺ is isovalent; La³⁺ replacement is aliovalent. The source workbook's tabulated Ti coefficient correlates with La, but the observed relation **Ti = 1 − La/2** is not direct evidence for physical Ti vacancies or a verified charge-compensation mechanism.

Formation energies are reported for calculated structures under the study's first-principles methodology (including VCA); the reported formation-energy target uses **eV/atom**. A DFT-labelled composition, an S1 candidate, an ML input vector and an experimentally fabricated capacitor represent different evidence levels.

**Phase 01 carried forward:** S1 has 30,301 candidates; S2–S4 each have 231 rows. S2–S3 and S2–S4 share consistent, explicitly keyed values in the published source audit. Only 21 S2 keys appeared in S1; 210 labelled keys were outside the S1 candidate list. These are source observations, not corrections to the publisher's methodology.

**Phase 02 outcome:** a typed field inventory and feature dictionary, a proposed relational schema, reproducibility checks, and an explicit register of unresolved scientific definitions.

## 2 | Source setup and execution

**Environment.** Python 3.11+ with Jupyter and `ipykernel`. Scientific computation uses **Python's standard library only**; no network access or package installation is required.

**Input.** `Supplementary Dataset.xlsx` or the same filename with a numbered suffix (for example, `Supplementary Dataset(3).xlsx`). Place the Excel file beside `BaTiO3_Phase02.ipynb` or under `data/raw/`. Set `SOURCE_OVERRIDE` only when an explicit local path is required.

**Execution.** Select an existing Python kernel in Jupyter, then choose **Kernel → Restart Kernel and Run All Cells**. The input workbook is never edited. A successful run creates `phase02_outputs/` beside the notebook. The selected source filename and SHA-256 provide provenance; the hash is not a guarantee of chemical validity.

**Output boundary.** No full S1–S4 data table, source row, original DOCX or publisher PDF is written to the outputs. Dictionaries reproduce original column headings and concise field descriptions for *private review*. These column-level materials are **not cleared for public GitHub publication** until reuse and licensing terms are checked.

**Execution progress.** On the first run, the large S1 worksheet is scanned in batches;
the notebook prints a line every 5,000 rows. This is normal, not a frozen kernel.
A private metadata-only cache in `phase02_outputs/.cache/` speeds up subsequent
**Restart Kernel and Run All Cells** executions. The cache is reused **only** when
its SHA-256 source fingerprint matches the workbook; otherwise the workbook is
rescanned. No raw spreadsheet measurement rows are copied into the cache.

**Local source location.** The workbook must be in the notebook's Jupyter working
folder (or `data/raw/`); the run prints `WORKING FOLDER` and `FOUND SOURCE` for
troubleshooting. If the workbook is elsewhere, set the optional `SOURCE_OVERRIDE`
in the setup code cell to its actual local `.xlsx` path. Do not publish that
private path on GitHub.


In [4]:
"""Phase 02 | Standard-library scientific schema and dictionary workflow.
Input workbook remains read-only. Exports contain no source measurement rows.
"""
from __future__ import annotations
import csv
from collections import Counter
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile, BadZipFile
from xml.etree import ElementTree as ET
import json
import re
import os
import tempfile
from posixpath import normpath

SHEETS = ("S1", "S2", "S3", "S4")
CHEM = ("Ba", "Sr", "La", "Ti", "O")
KEY_FIELDS = ("Ba", "Sr", "La", "Ti")
ENERGY = ("Final energy", "formation energy")
SOURCE_NAME = "Supplementary Dataset.xlsx"
SOURCE_OVERRIDE = None  # Optional local full path; keep private paths out of public GitHub
OUTPUT_DIR = Path.cwd() / "phase02_outputs"
# Private, checksum-validated cache of workbook metadata ONLY (never source data rows).
# The first run scans the full XLSX; subsequent Restart Kernel and Run All executions
# reuse this cache when the source file has exactly the same SHA-256 fingerprint.
CACHE_PATH = OUTPUT_DIR / ".cache" / "workbook_scan_v1.json"
CACHE_VERSION = "phase02_source_scan_v1"
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
REL = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
PKG = "{http://schemas.openxmlformats.org/package/2006/relationships}"

# Source Figure S1 lists 16 excluded constant features, in source order.
CONSTANT_FEATURES = (
    'r_X', 'Me_A', 'NoMe_A', 'NoMe_B', 'Mell_A', 'Mell_B',
    'N_VS_A', 'N_VP_A', 'N_VP_B', 'N_VF_A', 'N_VF_B',
    'UN_VS_A', 'UN_VS_B', 'UN_VP_A', 'UN_VF_A', 'N_OSE_A'
)

# Condensed scientific glossary from Supplementary information Table S1.
# The glossary is an interpretation aid, not a claim that units or aggregation are specified.
GLOSSARY = {
 'AN':'Atomic number', 'AW':'Atomic weight', 'Per':'Periodic-table period',
 'GP':'Periodic-table group number', 'Fam':'Element family category',
 'Me':'Metal indicator', 'NoMe':'Non-metal indicator', 'Mell':'Metalloid indicator',
 'MN':'Mendeleev number', 'LQN':'Orbital angular-momentum quantum number',
 'AR':'Atomic radius', 'Cov_R':'Covalent radius', 'ZRS':'Zunger radii sum',
 'IR':'Ionic radius', 'Cry_R':'Crystal radius',
 'PA_EN':'Pauling electronegativity', 'MB_EN':'Martynov–Batsanov electronegativity',
 'AR_EN':'Allred–Rochow electronegativity',
 'N_VE':'Number of valence electrons', 'N_VS':'Number of s valence electrons',
 'N_VP':'Number of p valence electrons', 'N_VD':'Number of d valence electrons',
 'N_VF':'Number of f valence electrons',
 'UN_VS':'Unfilled s valence-electron count', 'UN_VP':'Unfilled p valence-electron count',
 'UN_VD':'Unfilled d valence-electron count', 'UN_VF':'Unfilled f valence-electron count',
 'N_OSE':'Valence-electron category', '1st_IP':'First ionisation potential',
 'P':'Polarizability', 'MP':'Melting point', 'BP':'Boiling point',
 'Dens':'Density', 'SH':'Specific heat', 'HF':'Heat of fusion',
 'HV':'Heat of vaporisation', 'TC':'Thermal conductivity', 'Coh_E':'Cohesive energy',
 'Form_EN':'Formation energy'
}

### Code stage 2A — Resolve the original workbook

**Method.** Source discovery recognises numbered downloaded copies, checks for ambiguous files within the same directory and records a SHA-256 fingerprint. It does not silently select one of several equally placed workbooks. **Output:** one read-only input path; no source-data modification.

In [6]:
def file_key(name):
    p = Path(name)
    stem = re.sub(r'\s*\(\d+\)$', '', p.stem.strip())
    return re.sub(r'[^a-z0-9]', '', stem.casefold()) + p.suffix.casefold()

def find_source(folder, explicit=None):
    if explicit is not None and str(explicit).strip():
        src = Path(explicit).expanduser().resolve()
        if not src.is_file() or src.suffix.lower() != '.xlsx':
            raise FileNotFoundError('Configured workbook must exist and have .xlsx extension')
        return src
    root = Path(folder).resolve()
    search = [root, root / 'data' / 'raw', root.parent / 'data' / 'raw', root.parent]
    for directory in dict.fromkeys(search):
        if not directory.is_dir():
            continue
        matches = sorted((p for p in directory.iterdir() if p.is_file()
                          and file_key(p.name)==file_key(SOURCE_NAME)),
                         key=lambda p:p.name.casefold())
        if len(matches)>1:
            raise ValueError('AMBIGUOUS SOURCE: multiple matching workbooks in '+str(directory)
                             +'. Set SOURCE_OVERRIDE to a single verified file.')
        if matches:
            return matches[0]
    raise FileNotFoundError('SOURCE REQUIRED: place Supplementary Dataset.xlsx beside the '
                            'notebook or in data/raw (numbered suffixes accepted).')

def digest_file(path):
    digest = sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def excel_col(index):
    result = ''
    n = index + 1
    while n:
        n, rem = divmod(n-1, 26)
        result = chr(65+rem) + result
    return result

def col_index(ref):
    matched = re.match(r'([A-Za-z]+)', ref or '')
    if matched is None:
        raise ValueError(f'Invalid spreadsheet reference: {ref!r}')
    n=0
    for char in matched.group(1).upper(): n=n*26+(ord(char)-ord('A')+1)
    return n-1

## 3 | Column evidence: Excel provenance and field types

The workbook is inspected as a ZIP/OOXML file. Source headings, Excel coordinates and value categories are extracted without opening or resaving it in Excel. The parser streams one row at a time, which is important because S1 contains over 30,000 records. An Excel formula and a cached result are distinct: cached values are not interpreted as newly recalculated results.

**Inferred types are structural observations**, not declarations of physical units or scientific meaning. Nullability, allowed ranges and conversion rules are recorded separately rather than guessed from familiar field names.

In [8]:
def shared_strings(z):
    if 'xl/sharedStrings.xml' not in z.namelist():
        return []
    words=[]
    with z.open('xl/sharedStrings.xml') as stream:
        for _, element in ET.iterparse(stream, events=('end',)):
            if element.tag == NS+'si':
                words.append(''.join(node.text or '' for node in element.iter(NS+'t')))
                element.clear()
    return words

def sheet_paths(z):
    if 'xl/workbook.xml' not in z.namelist() or 'xl/_rels/workbook.xml.rels' not in z.namelist():
        raise ValueError('INVALID XLSX: missing workbook/index XML')
    book = ET.fromstring(z.read('xl/workbook.xml'))
    rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
    refs={item.get('Id'):item.get('Target') for item in rels.findall(PKG+'Relationship')}
    output={}
    for item in book.findall('./'+NS+'sheets/'+NS+'sheet'):
        name, rid=item.get('name'),item.get(REL+'id')
        if rid not in refs: raise ValueError('INVALID XLSX: broken sheet relationship')
        target=refs[rid].replace('\\','/')
        if target.startswith('/'): f=target.lstrip('/')
        elif target.startswith('xl/'): f=target
        else: f=normpath('xl/'+target)
        if f not in z.namelist(): raise ValueError('INVALID XLSX: missing worksheet '+name)
        output[name]=f
    if set(SHEETS)-set(output):
        raise ValueError('MISSING SOURCE SHEETS: '+', '.join(sorted(set(SHEETS)-set(output))))
    return output

def parse_cell(el, strings):
    v=el.find(NS+'v'); f=el.find(NS+'f')
    raw=v.text if v is not None else None
    typ=el.get('t', 'n')
    if typ=='s' and raw is not None:
        try: raw=strings[int(raw)]
        except (ValueError, IndexError): raise ValueError('Broken shared-string index')
    elif typ=='inlineStr': raw=''.join(t.text or '' for t in el.iter(NS+'t'))
    return raw, f is not None, typ

def stream_rows(z, path, strings):
    with z.open(path) as stream:
        events=ET.iterparse(stream, events=('start','end'))
        _, root = next(events)
        for event, row in events:
            if event!='end' or row.tag!=NS+'row': continue
            cells={}
            for el in row.iter(NS+'c'):
                cells[col_index(el.get('r'))]=parse_cell(el, strings)
            yield int(row.get('r','0')), cells
            row.clear(); root.clear()

def inspect_source(path, preverified_sha256=None):
    workbook={'sheets':{},'source_filename':path.name,'source_size_bytes':path.stat().st_size,
              'source_sha256':preverified_sha256 or digest_file(path)}
    try:
        with ZipFile(path) as z:
            names=sheet_paths(z); strings=shared_strings(z)
            for sheet in SHEETS:
                iterator=stream_rows(z,names[sheet],strings)
                try: _, head_cells=next(iterator)
                except StopIteration: raise ValueError('EMPTY WORKSHEET: '+sheet)
                indexes=sorted(head_cells)
                if indexes!=list(range(len(indexes))):
                    raise ValueError('HEADER GAP: '+sheet)
                headers=[head_cells[i][0] for i in indexes]
                if any(not h or not str(h).strip() for h in headers):
                    raise ValueError('EMPTY SOURCE HEADER: '+sheet)
                if len(set(headers))!=len(headers):
                    raise ValueError('DUPLICATED SOURCE HEADER: '+sheet)
                print(f'SCANNING {sheet}: {len(headers)} source fields', flush=True)
                stats=[Counter() for _ in headers]
                data_rows=0
                for excel_row, cells in iterator:
                    data_rows+=1
                    if data_rows % 5000 == 0:
                        print(f'  {sheet}: {data_rows:,} rows scanned...', flush=True)
                    for j in range(len(headers)):
                        value, formula, typ=cells.get(j,(None,False,'n'))
                        if value is None or str(value).strip()=='':
                            stats[j]['empty']+=1
                        elif typ in ('s','inlineStr','str'):
                            stats[j]['text']+=1
                        elif typ=='b': stats[j]['boolean']+=1
                        elif typ=='e': stats[j]['error']+=1
                        else:
                            try:
                                number=float(value)
                                if number!=number or number in (float('inf'),float('-inf')):
                                    stats[j]['invalid_numeric']+=1
                                elif number.is_integer():stats[j]['integer_like']+=1
                                else:stats[j]['numeric']+=1
                            except (ValueError,TypeError,OverflowError):stats[j]['other']+=1
                        if formula:stats[j]['formula']+=1
                print(f'COMPLETE {sheet}: {data_rows:,} rows',flush=True)
                workbook['sheets'][sheet]={'headers':headers,'data_rows':data_rows,
                                            'observed_types':[dict(c) for c in stats]}
    except BadZipFile as err:
        raise ValueError('INVALID XLSX: damaged archive') from err
    return workbook

def cached_workbook(path, input_sha):
    """Use verified schema/type metadata only if the exact source bytes match."""
    if not CACHE_PATH.is_file():
        return None
    try:
        cached = json.loads(CACHE_PATH.read_text(encoding='utf-8'))
        if (cached.get('cache_version') != CACHE_VERSION or
                cached.get('source_sha256') != input_sha or
                cached.get('source_size_bytes') != path.stat().st_size):
            return None
        workbook = cached.get('workbook')
        if not isinstance(workbook, dict) or set(workbook.get('sheets',{})) != set(SHEETS):
            return None
        for sheet in SHEETS:
            record = workbook['sheets'][sheet]
            if (not record.get('headers') or record.get('data_rows',0) <= 0 or
                    len(record['headers']) != len(record.get('observed_types',[]))):
                return None
        # The same workbook may have a different numbered-download filename.
        workbook['source_filename'] = path.name
        print('CACHE HIT: identical SHA-256 source; full XML rescan not needed.',flush=True)
        return workbook
    except (OSError, ValueError, TypeError, KeyError, json.JSONDecodeError):
        print('CACHE WARNING: unreadable metadata cache; rebuilding from original workbook.', flush=True)
        return None


def save_scan_cache(workbook):
    """Save only per-field audit statistics and headings; never raw spreadsheet rows."""
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'cache_version':CACHE_VERSION,
        'source_sha256':workbook['source_sha256'],
        'source_size_bytes':workbook['source_size_bytes'],
        'workbook':workbook,
    }
    with tempfile.TemporaryDirectory(prefix='scan_cache_',dir=CACHE_PATH.parent) as tempdir:
        staged = Path(tempdir) / CACHE_PATH.name
        staged.write_text(json.dumps(payload,ensure_ascii=False,separators=(',', ':'))+'\n',encoding='utf-8')
        os.replace(staged,CACHE_PATH)


## 4 | Field definitions and scientific units

Each original heading is assigned a **source role** before any value is transformed. `NO.` is a worksheet identifier, Ba/Sr/La/Ti/O are tabulated coefficients, S4 `Method` and `Space Group` are reported calculation/structure labels, and the two energy fields represent distinct quantities. `formation energy` is identified as the reported DFT target in **eV/atom**. The available supplementary text does not establish a consistent unit for every elemental descriptor or for `Final energy`; these are marked **not verified**, not silently filled in.

Supplementary information **Table S1** describes common abbreviations, including `AN`, `AW`, `IR`, `PA_EN`, `N_VE` and `Coh_E`. A suffix such as `_A` or `_B` denotes a nominal crystal-site feature, **not** proof of a particular site-averaging or La-substitution formula. Unexplained features (for example `t1`, `t2`, `OF`, `Gou`, and `r_A` as an exact formula) retain an explicit review status.

In [10]:
def interpretation(header):
    if header=='NO.':
        return ('source_row_label','record','Original worksheet record label','not applicable','verified role')
    if header in CHEM:
        site='A' if header in ('Ba','Sr','La') else ('B' if header=='Ti' else 'X')
        return ('source_coefficient',site,
                'Tabulated '+header+' composition coefficient; source encoding retained',
                'dimensionless (source coefficient)','verified role; chemistry pending review')
    if header=='Method':
        return ('calculation_method','calculation','Reported computational method label',
                'not applicable','verified source role')
    if header=='Space Group':
        return ('structure_label','structure','Reported crystallographic space-group label',
                'not applicable','verified source role; no new structure validation')
    if header=='formation energy':
        return ('reported_DFT_target','calculation','Published formation-energy target',
                'eV/atom','reported unit; definition in supplementary Text S1')
    if header=='Final energy':
        return ('reported_final_energy','calculation','Reported final energy (distinct from formation-energy target)',
                'unit not independently verified','unit requires review')
    for suffix,site in (('_A','A'),('_B','B'),('_X','X')):
        if header.endswith(suffix):
            base=header[:-len(suffix)]
            if base in GLOSSARY:
                return ('elemental_descriptor',site,GLOSSARY[base]+'; site-labelled source feature',
                        'source unit not verified','abbreviation supported by Supplementary Table S1; method/units pending review')
            return ('derived_descriptor',site,
                    'Source descriptor '+header+'; exact physical definition not verified',
                    'source unit not verified','scientific definition pending review')
    if header in ('t1','t2','OF') or header.startswith('Gou'):
        return ('derived_descriptor','composition',
                'Derived numerical descriptor; exact equation/aggregation not confirmed',
                'source unit not verified','scientific definition pending review')
    if header in GLOSSARY:
        return ('descriptor','composition',GLOSSARY[header],
                'source unit not verified','aggregation/units pending review')
    return ('derived_descriptor','unspecified',
            'Workbook descriptor; definition requires source-method reconciliation',
            'source unit not verified','scientific definition pending review')

def observed_type(counter):
    present=[n for n in ('integer_like','numeric','text','boolean','error','invalid_numeric','other')
             if counter.get(n,0)]
    if not present:return 'all_missing'
    if set(present) <= {'integer_like','numeric'}:return 'numeric'
    if present==['text']:return 'text'
    if present==['boolean']:return 'boolean'
    return 'mixed_or_invalid:'+','.join(present)

def produce_dictionaries(wb):
    inventory=[]
    for sheet in SHEETS:
        data=wb['sheets'][sheet]
        for j,(header,counts) in enumerate(zip(data['headers'],data['observed_types'])):
            role,site,meaning,unit,interpretation_status=interpretation(header)
            inventory.append({
              'sheet':sheet,'excel_column':excel_col(j),'source_header':header,
              'field_role':role,'site_or_entity':site,'definition':meaning,
              'observed_value_type':observed_type(counts),'unit_policy':unit,
              'definition_status':interpretation_status,
              'nonempty_count':data['data_rows']-counts.get('empty',0),
              'empty_count':counts.get('empty',0),'formula_cells':counts.get('formula',0),
              'missingness_policy':'Preserve missing; do not infer zero or unsuccessful calculation',
              'source_pointer':sheet+'!'+excel_col(j)+'1',
              'next_action':('Check source scientific definition or unit' if 'pending' in interpretation_status
                             or 'requires review' in interpretation_status or 'not verified' in unit else
                             'Keep source provenance; validate in Phase 03/04')
            })
    h2=wb['sheets']['S2']['headers']; h3=wb['sheets']['S3']['headers']
    s2pos={name:excel_col(i) for i,name in enumerate(h2)}
    s1pos={name:excel_col(i) for i,name in enumerate(wb['sheets']['S1']['headers'])}
    dictionary=[]
    for i,h in enumerate(h3):
        role,site,meaning,unit,status=interpretation(h)
        dictionary.append({
          'source_model_input_order':i+1,'source_header':h,
          'input_class':'composition coefficient' if h in KEY_FIELDS else 'computed / elemental descriptor',
          'site_or_entity':site,'scientific_description':meaning,'unit_policy':unit,
          'definition_status':status,'S3_reference':'S3!'+excel_col(i)+'1',
          'S2_reference':'S2!'+s2pos[h]+'1' if h in s2pos else 'MISSING',
          'S1_reference':'S1!'+s1pos[h]+'1' if h in s1pos else 'NOT FOUND',
          'missing_value_policy':'Do not impute in Phase 02',
          'interpretation_action':'Confirm aggregation, units and method where undocumented'
        })
    return inventory,dictionary

## 5 | The 100-to-82 feature-accounting question

The original workbook contains **102 columns in S2** and **82 columns in S3**. The source supplementary figure reports removal of 16 constant features, one descriptive `Final energy` column and one `formation energy` target. The two additional S2-only columns are `NO.` and `O`, which are not ML input columns. This produces a **workbook-level reconciliation**:


**Header reconciliation:** S2 has 102 total columns; excluding the record index and O coefficient gives a 100-field comparison set.  
Removing 16 reported constants and two energy fields yields the 82 S3 columns.  
S3 includes **four chemistry coefficients (Ba, Sr, La, Ti)** and **78 additional descriptors**.

This is a transparent accounting convention inferred from the provided headers and source feature-screening statement; it does not independently reproduce the study's feature-engineering algorithms.

In [12]:
def reconcile_features(wb):
    s1=wb['sheets']['S1']['headers'];s2=wb['sheets']['S2']['headers'];s3=wb['sheets']['S3']['headers']
    s2_not_selected=[h for h in s2 if h not in set(s3)]
    expected=set(CONSTANT_FEATURES)|set(ENERGY)|{'NO.','O'}
    unknown=set(s2_not_selected)-expected
    not_present=expected-set(s2_not_selected)
    return {
      'S1_total_source_columns':len(s1), 'S2_total_source_columns':len(s2),
      'S3_input_columns':len(s3),'S4_total_source_columns':len(wb['sheets']['S4']['headers']),
      'S3_composition_coefficients':sum(h in KEY_FIELDS for h in s3),
      'S3_noncomposition_descriptors':sum(h not in KEY_FIELDS for h in s3),
      'S2_accounting_base_excluding_NO_and_O':len([h for h in s2 if h not in ('NO.','O')]),
      'reported_constant_feature_count':len(CONSTANT_FEATURES),
      'constant_features_all_excluded':set(CONSTANT_FEATURES).issubset(s2_not_selected),
      'missing_S3_headers_in_S2':sorted(set(s3)-set(s2)),
      'unexpected_excluded_fields':sorted(unknown),
      'expected_exclusions_not_found':sorted(not_present),
      'S2_exclusion_list':s2_not_selected,
      'feature_reconciliation_status':('RECONCILED_HEADER_ACCOUNTING'
         if len(s2)==102 and len(s3)==82 and not unknown and not not_present
         and set(s3).issubset(set(s2)) and len(CONSTANT_FEATURES)==16 else 'REVIEW_REQUIRED')
    }

## 6 | Proposed evidence-linked schema

Phase 02 separates six core entities: **source_record**, **composition**, **calculation**, **descriptor_definition**, **feature_vector** and **qc_finding**. A `composition_id` references tabulated coefficients but cannot substitute for a validated crystallographic or experimental specimen identity. Each calculation has its own method, target definition and unit; the feature vector has its own source and selection version. Proposed physical specimen and experimental-measurement tables remain unpopulated because no such records are provided in this workbook.

**Key rules.** Unknown is not zero; an unlabelled candidate is not a failed calculation; matching workbook rows are not an adequate linkage criterion; and a chemical representation anomaly is not silently repaired. Source-provenance keys retain the sheet and original coordinate. Later phases perform full composition consistency tests and cross-sheet record joining.

In [14]:
def build_schema_markdown(wb, counts):
    sections=[
      '# Phase 02 — Scientific data schema', '',
      '## Status and purpose', '',
      'Proposed, versioned logical schema for the published BaTiO3 computational workbook. '
      'The input workbook remains unchanged. This is a data model, not a populated experimental database.', '',
      '## Source identity', '',
      f"- Source: `{wb['source_filename']}`",f"- SHA-256: `{wb['source_sha256']}`", '',
      '## Entity design', '',
      '| Entity | Primary key / linkage | Essential fields | Evidence meaning |',
      '|---|---|---|---|',
      '| `source_record` | `source_record_id` | source hash, sheet, row, column, header, original value reference | Immutable origin pointer; raw values remain in local workbook |',
      '| `composition` | `composition_id` | Ba, Sr, La, Ti, O source coefficients; nominal formula; representation review | Computational composition identity, not specimen identity |',
      '| `calculation` | `calculation_id`, `composition_id` | reported method, space-group label, energy name, source unit, source pointer | Reported DFT calculation evidence |',
      '| `descriptor_definition` | `descriptor_id` | name, site, source abbreviation, physical meaning, unit status | Meaning of a candidate model feature |',
      '| `feature_vector` | `feature_vector_id`, `composition_id` | S3 field names, descriptor version, feature linkage source | Model input representation; no model output implied |',
      '| `qc_finding` | `finding_id`, optional record key | issue, severity, source location, interpretation, disposition | Traceable review, never silent correction |',
      '| `experimental_specimen` (future) | `specimen_id` | provenance, batch, processing, test protocol | Not populated by current data |', '',
      '## Evidence and missingness policies', '',
      '- Preserve candidate-only, reported-DFT, ML-predicted, experimentally measured, documented failure and unknown as distinct categories.',
      '- Preserve source coefficients before constructing nominal formulas or any chemically interpreted correction.',
      '- A missing value is unknown and never automatically a failed synthesis or calculation.',
      '- The reported formation-energy target is in eV/atom; other descriptor units require a separate published definition or manual verification.',
      '- No unverified Ti vacancy or charge-compensation mechanism is inferred from Ti = 1 - La/2.',
      '- Record join keys are chemistry-aware and evidence-linked; row position is not an identity.', '',
      '## Workbook field-accounting observations', '',
      f"- S1: {wb['sheets']['S1']['data_rows']:,} source rows and {len(wb['sheets']['S1']['headers'])} columns.",
      f"- S2: {wb['sheets']['S2']['data_rows']:,} source rows and {len(wb['sheets']['S2']['headers'])} columns.",
      f"- S3: {wb['sheets']['S3']['data_rows']:,} source rows and {len(wb['sheets']['S3']['headers'])} columns.",
      f"- S4: {wb['sheets']['S4']['data_rows']:,} source rows and {len(wb['sheets']['S4']['headers'])} columns.",
      f"- S3 includes {counts['S3_composition_coefficients']} composition coefficients and {counts['S3_noncomposition_descriptors']} other descriptors.",
      f"- Header accounting: {counts['feature_reconciliation_status']}; this is not independent recalculation of the published feature construction.", '',
      '## Review items and phase boundary', '',
      '- Ti–La source coefficient representation and its relation to VCA assumptions remain unresolved.',
      '- The source S1 candidate space does not contain all S2 labelled keys (Phase 01 finding).',
      '- Unit and aggregation definitions are not uniformly supplied for 82 input fields; uncertain entries are explicitly marked.',
      '- Phase 03: chemical consistency; Phase 04: standardisation and feature definitions; Phase 05: verified record linkage.', '',
      '## Attribution and redistribution', '',
      '- Article: https://doi.org/10.1063/5.0314819',
      '- Official supplementary collection: https://doi.org/10.60893/figshare.jap.c.8234602',
      '- Source data, full workbook, supplementary DOCX and publisher figures are not included. '
      'Column-level metadata and derivative CSV files require a separate licensing review before public redistribution.', ''
    ]
    return '\n'.join(sections)

## 7 | Validation checks, deterministic exports and research decisions

The automated checks verify source sheet inventory, essential columns, the 82-field S3 selection, feature exclusion bookkeeping, data types, missingness and required scientific role declarations. Checks that fail prevent an unqualified `COMPLETE` status. Open scientific questions, even when all structural checks pass, produce **`COMPLETED_WITH_REVIEW_ITEMS`**.

**Generated locally:** `schema_design.md`, `data_dictionary.csv`, `descriptor_dictionary.csv`, `feature_accounting.json`, `scientific_review_items.csv`, `phase02_summary.md` and `run_manifest.json`. The CSV dictionaries include *publisher-derived headings*: they are useful working products but should not be put on public GitHub without a source licence review.

In [16]:
def self_tests():
    tests=[]
    def check(name, statement):
        if not statement:raise AssertionError('ENGINEERING TEST FAILED: '+name)
        tests.append(name)
    check('Numbered source filename accepted',file_key('Supplementary Dataset(3).xlsx')==file_key(SOURCE_NAME))
    check('Other source rejected',file_key('another.xlsx')!=file_key(SOURCE_NAME))
    check('Excel A',excel_col(0)=='A')
    check('Excel CX',excel_col(101)=='CX')
    check('Index CX',col_index('CX17')==101)
    check('All four sheets declared',len(SHEETS)==4)
    check('16 constant features',len(CONSTANT_FEATURES)==16 and len(set(CONSTANT_FEATURES))==16)
    check('Formation-energy unit declared',interpretation('formation energy')[3]=='eV/atom')
    check('Final-energy unit stays uncertain','verified' in interpretation('Final energy')[3])
    check('Composition coefficients separated from descriptors',interpretation('La')[0]=='source_coefficient')
    check('Ti role at B site',interpretation('Ti')[1]=='B')
    check('Unknown descriptor not fabricated','pending review' in interpretation('t1')[4])
    check('Known abbreviation supported','Atomic number' in interpretation('AN_A')[2])
    check('Exclusions have no target leakage',not set(ENERGY).intersection(KEY_FIELDS))
    check('No external source row export function',callable(produce_dictionaries))
    return tests

def validate_source(wb, dictionary, accounting):
    findings=[]
    def finding(severity, code, evidence, action):
        findings.append({'severity':severity,'code':code,'evidence':evidence,'review_action':action})
    for sheet in SHEETS:
        data=wb['sheets'].get(sheet,{})
        if not data.get('headers'):finding('ERROR','NO_HEADERS',sheet,'Examine workbook structure')
        if data.get('data_rows',0)<=0:finding('ERROR','NO_DATA_ROWS',sheet,'Verify source worksheet')
        for key in KEY_FIELDS:
            if key not in data.get('headers',[]):
                finding('ERROR','MISSING_COMPOSITION_FIELD',sheet+':'+key,'Verify source sheet identity')
    for h in ENERGY:
        if h not in wb['sheets']['S2']['headers'] or h not in wb['sheets']['S4']['headers']:
            finding('ERROR','MISSING_ENERGY_FIELD',h,'Recheck S2/S4 source fields')
    if len(dictionary)!=82:
        finding('ERROR','INPUT_WIDTH_UNEXPECTED',str(len(dictionary)),'Compare workbook with source publication')
    if accounting['feature_reconciliation_status']!='RECONCILED_HEADER_ACCOUNTING':
        finding('ERROR','FEATURE_ACCOUNTING_CONFLICT',str(accounting),'Review original feature list')
    if any(row['observed_value_type'].startswith('mixed_or_invalid') for row in DATA_INVENTORY):
        finding('REVIEW','MIXED_TYPES','At least one source field has mixed/non-numeric values',
                'Review field-level type counts; preserve original source values')
    finding('REVIEW','TI_LA_SOURCE_REPRESENTATION',
            'Phase 01 found 210/231 labelled records with Ti below 1 and 231/231 matching Ti = 1 - La/2',
            'Reconcile VCA and source-encoding assumptions; no automatic Ti correction')
    finding('REVIEW','S1_S2_POPULATION_MISMATCH',
            'Phase 01 found only 21 of 231 S2 chemistry keys represented in S1 candidate list',
            'Carry into Phase 05 evidence linkage; do not equate population coverage')
    pending=sum('pending review' in item['definition_status'] or
                'not verified' in item['unit_policy'] for item in dictionary
                if item['source_header'] not in KEY_FIELDS)
    if pending:
        finding('REVIEW','DESCRIPTOR_SEMANTICS_INCOMPLETE',
                f'{pending} of {len(dictionary)} S3 non-composition descriptor fields need unit or interpretation review',
                'Confirm equations, site aggregation and units against the authors’ methods before scientific reuse')
    return findings

def write_csv(path,rows):
    if not rows:raise ValueError('Cannot write empty scientific dictionary: '+path.name)
    with path.open('w',encoding='utf-8',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0].keys()),lineterminator='\n')
        writer.writeheader();writer.writerows(rows)

def write_files(folder,wb,inventory,dictionary,accounting,findings,test_names):
    folder.mkdir(parents=True,exist_ok=True)
    # Files staged locally then atomically replaced to avoid incomplete exports.
    with tempfile.TemporaryDirectory(prefix='phase02_',dir=folder.parent) as temp:
        temp=Path(temp)
        write_csv(temp/'data_dictionary.csv',inventory)
        write_csv(temp/'descriptor_dictionary.csv',dictionary)
        write_csv(temp/'scientific_review_items.csv',findings)
        (temp/'schema_design.md').write_text(build_schema_markdown(wb,accounting),encoding='utf-8')
        (temp/'feature_accounting.json').write_text(json.dumps(accounting,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        blocking=[f for f in findings if f['severity']=='ERROR']
        status='BLOCKED_STRUCTURAL_ERRORS' if blocking else ('COMPLETED_WITH_REVIEW_ITEMS' if findings else 'COMPLETE')
        summary=[
          '# Phase 02 — Scientific schema and data dictionary', '',
          '## Result', '',f'**Status:** `{status}`', '',
          f"Source workbook SHA-256: `{wb['source_sha256']}`",'',
          '## Evidence-based results', '',
          f"- Source columns inventoried: **{len(inventory)}** across four worksheets.",
          f"- S3 ML input columns: **{len(dictionary)}** = **{accounting['S3_composition_coefficients']}** composition coefficients + **{accounting['S3_noncomposition_descriptors']}** other descriptors.",
          f"- Reconciliation of S2 102 columns into the reported 100-to-82 fields: **{accounting['feature_reconciliation_status']}**.",
          f"- Engineering self-tests: **{len(test_names)} passed**.",
          f"- Recorded scientific review findings: **{len(findings)}** (including {len(blocking)} blocking errors).", '',
          '## Scientific interpretation', '',
          'The schema identifies source fields and evidence classes while preserving their original definitions. '
          'It does not assert that units and site-weighting formulas have been verified for all descriptor names. '
          'DFT formation energy (eV/atom) is distinct from the source Final energy column, whose unit was not independently verified.', '',
          '## Pending questions', ''
        ]
        for f in findings:summary.append(f"- **{f['code']}** ({f['severity']}): {f['review_action']}.")
        summary.extend(['', '## Publication note', '',
          'These files are **private working outputs**, not an approved GitHub release. '
          'Confirm the applicable licence for publisher-derived column headings and supplementary metadata; '
          'retain original supplementary files only locally until their reuse rights are verified.', '',
          '## Next phase', '',
          'Phase 03 will implement chemical consistency rules on the tabulated coefficients without rewriting the original workbook.', ''])
        (temp/'phase02_summary.md').write_text('\n'.join(summary),encoding='utf-8')
        # Output digests are deterministic. No local file system paths or user identifiers in manifest.
        output_hashes={p.name:digest_file(p) for p in sorted(temp.iterdir()) if p.is_file()}
        manifest={
          'phase':'02','status':status,'input_name':wb['source_filename'],
          'input_sha256':wb['source_sha256'],'input_bytes':wb['source_size_bytes'],
          'source_modified':False,'sheet_counts':{k:{'rows':wb['sheets'][k]['data_rows'],
                   'columns':len(wb['sheets'][k]['headers'])} for k in SHEETS},
          'engineering_tests_passed':len(test_names),'review_findings':len(findings),
          'generated_file_sha256':output_hashes,'redistribution':'PENDING_LICENSE_REVIEW'}
        (temp/'run_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        for src in sorted(temp.iterdir()):os.replace(src,folder/src.name)
    return status, {p.name:digest_file(p) for p in sorted(folder.iterdir()) if p.is_file()}

## 8 | Restart-safe execution and run record

The self-tests run before the workbook scan. A missing or ambiguous workbook produces an explicit input error instead of silently falling back to old output files. Results are written after validation, and the source hash is checked again to detect changes during the run. The final status preserves unresolved scientific questions as review items, even when the code completes successfully.

The full workbook scan may be slow during the first execution, particularly on
Windows. Progress messages distinguish active XML processing from a crash.
Repeat runs use a checksum-validated metadata cache, but all dictionary and
scientific-review checks still execute on each run.


In [18]:
import time
RUN_STATUS='NOT_STARTED'
RUN_ERROR=None
SOURCE=None
WORKBOOK=None
DATA_INVENTORY=[]
DESCRIPTOR_DICTIONARY=[]
ACCOUNTING={}
FINDINGS=[]
TESTS=[]
OUTPUT_HASHES={}
try:
    TESTS=self_tests()
    print(f'ENGINEERING TESTS: PASS ({len(TESTS)} checks)')
    print('WORKING FOLDER:',Path.cwd(),flush=True)
    SOURCE=find_source(Path.cwd(),explicit=SOURCE_OVERRIDE)
    print('FOUND SOURCE:',SOURCE.name,'| size:',f'{SOURCE.stat().st_size/1024/1024:.1f} MiB',flush=True)
    started=time.perf_counter()
    source_sha=digest_file(SOURCE)
    WORKBOOK=cached_workbook(SOURCE,source_sha)
    if WORKBOOK is None:
        print('CACHE MISS: scanning original workbook; S1 is the largest sheet.',flush=True)
        WORKBOOK=inspect_source(SOURCE,preverified_sha256=source_sha)
    print(f'WORKBOOK METADATA READY: {time.perf_counter()-started:.1f} s',flush=True)
    print('SOURCE:',SOURCE.name,'| SHA-256:',WORKBOOK['source_sha256'][:16]+'…')
    DATA_INVENTORY, DESCRIPTOR_DICTIONARY=produce_dictionaries(WORKBOOK)
    ACCOUNTING=reconcile_features(WORKBOOK)
    FINDINGS=validate_source(WORKBOOK,DESCRIPTOR_DICTIONARY,ACCOUNTING)
    if digest_file(SOURCE)!=WORKBOOK['source_sha256']:
        raise RuntimeError('SOURCE CHANGED DURING SCAN: outputs not valid')
    RUN_STATUS,OUTPUT_HASHES=write_files(OUTPUT_DIR,WORKBOOK,DATA_INVENTORY,
                                        DESCRIPTOR_DICTIONARY,ACCOUNTING,FINDINGS,TESTS)
    try:
        save_scan_cache(WORKBOOK)
    except OSError as cache_error:
        # Cache is optional; no scientific output is invalidated if it cannot be saved.
        print('CACHE NOTE: could not save optional acceleration cache:',cache_error,flush=True)
    print('OUTPUT STATUS:',RUN_STATUS,'| files:',len(OUTPUT_HASHES),flush=True)
except (FileNotFoundError,ValueError,OSError,BadZipFile,AssertionError,RuntimeError) as exc:
    RUN_STATUS='NOT_COMPLETED'
    RUN_ERROR=f'{type(exc).__name__}: {exc}'
    print('PHASE 02 NOT COMPLETED:',RUN_ERROR)
    print('Previous outputs, if present, are not evidence of this run.')

ENGINEERING TESTS: PASS (15 checks)
WORKING FOLDER: C:\Users\sufer\Documents\BaTiO₃ Materials Data Curation_AI-Ready Workflow Engineering\BaTiO3_Phase02
FOUND SOURCE: Supplementary Dataset.xlsx | size: 25.9 MiB
CACHE HIT: identical SHA-256 source; full XML rescan not needed.
WORKBOOK METADATA READY: 0.1 s
SOURCE: Supplementary Dataset.xlsx | SHA-256: 5d28d5641ea8c564…
OUTPUT STATUS: COMPLETED_WITH_REVIEW_ITEMS | files: 7


WORKBOOK METADATA READY: 0.0 s


SOURCE: Supplementary Dataset(3).xlsx | SHA-256: 5d28d5641ea8c564…
OUTPUT STATUS: COMPLETED_WITH_REVIEW_ITEMS | files: 7


## 9 | Findings and follow-up decisions

The final status distinguishes **successful schema extraction** from **scientific interpretation that still needs source-method review**. Complete header accounting is useful evidence of feature-selection bookkeeping; it is not a validation of the publisher's DFT convergence, site-averaging methodology, physical defect chemistry or capacitor performance. Every unresolved item has a review action rather than an assumed correction.

In [20]:
print('\n'+'='*68)
print('PHASE 02 — FINAL STATUS:',RUN_STATUS)
print('='*68)
if WORKBOOK is not None:
    for sheet in SHEETS:
        x=WORKBOOK['sheets'][sheet]
        print(f"{sheet}: {x['data_rows']:,} source rows | {len(x['headers'])} fields")
    print('DATA DICTIONARY:',len(DATA_INVENTORY),'source fields')
    print('DESCRIPTOR DICTIONARY:',len(DESCRIPTOR_DICTIONARY),'S3 input fields')
    print('S3 composition fields:',ACCOUNTING['S3_composition_coefficients'],
          '| other descriptors:',ACCOUNTING['S3_noncomposition_descriptors'])
    print('100-to-82 accounting:',ACCOUNTING['feature_reconciliation_status'])
    print('SCIENTIFIC REVIEW ITEMS:',len(FINDINGS))
    for issue in FINDINGS:print(f"  {issue['severity']}: {issue['code']}")
    print('EXPORTED:',', '.join(sorted(OUTPUT_HASHES)))
if RUN_ERROR:print('ERROR:',RUN_ERROR)


PHASE 02 — FINAL STATUS: COMPLETED_WITH_REVIEW_ITEMS
S1: 30,301 source rows | 100 fields
S2: 231 source rows | 102 fields
S3: 231 source rows | 82 fields
S4: 231 source rows | 10 fields
DATA DICTIONARY: 294 source fields
DESCRIPTOR DICTIONARY: 82 S3 input fields
S3 composition fields: 4 | other descriptors: 78
100-to-82 accounting: RECONCILED_HEADER_ACCOUNTING
SCIENTIFIC REVIEW ITEMS: 3
  REVIEW: TI_LA_SOURCE_REPRESENTATION
  REVIEW: S1_S2_POPULATION_MISMATCH
  REVIEW: DESCRIPTOR_SEMANTICS_INCOMPLETE
EXPORTED: data_dictionary.csv, descriptor_dictionary.csv, feature_accounting.json, phase02_summary.md, run_manifest.json, schema_design.md, scientific_review_items.csv


## 10 | Scientific conclusion and release boundary

**Achieved when the final status is `COMPLETED_WITH_REVIEW_ITEMS`:** a read-only, worksheet-traceable field inventory; a defined 82-input scientific descriptor table; a transparent 100-to-82 header-accounting reconciliation; a logical materials-evidence schema; and runnable engineering checks with review actions. This adds field-level scientific memory to the Phase 01 source audit without reinterpreting or modifying raw measurements.

**Still unresolved:** the Ti–La coefficient convention, published source-population differences, missing descriptor equations and site-aggregation rules, and unverified units for several fields. These are material scientific questions, **not** failures to be concealed by arbitrary cleaning.

**Publication boundary:** neither the source workbook nor the supplementary DOCX is republished; dictionary CSVs containing publisher-derived headers are not automatically approved for public release. A separately sanitised GitHub notebook and a licence-reviewed derivative report should be created only after verifying redistribution permissions.

**Next scientific step:** Phase 03 — chemical consistency and data quality, using the schema from this phase without changing Phase 01 outputs or the original workbook.

**Runtime note.** A successful first run creates a private `.cache/` subfolder
under `phase02_outputs/`; it must not be included in any public GitHub upload.
If the source workbook changes, its SHA-256 mismatch automatically triggers a
fresh scan, preventing stale cached scientific metadata from being treated as
new evidence.
